<a href="https://colab.research.google.com/github/SyameimaruKoa/Minecraft-mod-AutoTranslation-tool/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft Mod Auto Translator — Google Colab（LM Link）

ColabでLM Studio（llmster）を動かし、LM Link経由でPCと接続するのじゃ。ログイン情報はGoogleドライブへ**AES-256-GCMで暗号化保存**する。PLaMoのGGUFは公式Hugging Faceから直接ダウンロードする。

## 1. LM Studioのインストール

先にインストールし、次のセルで認証情報を復元してからデーモンを起動する。

In [ ]:
import os
import subprocess
from pathlib import Path

subprocess.run(["pkill", "-f", "llmster"], check=False)
subprocess.run(["pkill", "-f", "lmstudio"], check=False)
subprocess.run(["bash", "-c", "curl -fsSL https://lmstudio.ai/install.sh | bash"], check=True)
os.environ["PATH"] = str(Path.home() / ".lmstudio" / "bin") + os.pathsep + os.environ["PATH"]
subprocess.run(["lms", "bootstrap"], check=True)
subprocess.run(["sync"], check=True)
print("LM Studioのセットアップが完了したのじゃ。")

## 2. Googleドライブの暗号化保管庫

Googleドライブのマウントを承認する。保管先は `マイドライブ/LMStudio/colab-lmlink-auth.v1.enc`。パスフレーズは表示・保存せず、入力時も非表示じゃ。毎回入力する代わりにColabの「シークレット」に `LM_LINK_VAULT_PASSWORD` を登録してこのノートブックへのアクセスを許可してもよい。**認証情報の復元には同じパスフレーズが必要**じゃ。

In [ ]:
import getpass
import hashlib
import io
import os
import secrets
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive, userdata

try:
    from cryptography.hazmat.primitives.ciphers.aead import AESGCM
except ImportError:
    env_dir = Path("/content/.lmlink-crypto-venv")
    subprocess.run([sys.executable, "-m", "venv", str(env_dir)], check=True)
    python = env_dir / "bin" / "python"
    subprocess.run([str(python), "-m", "pip", "install", "cryptography"], check=True)
    site_packages = next((env_dir / "lib").glob("python*/site-packages"))
    sys.path.insert(0, str(site_packages))
    from cryptography.hazmat.primitives.ciphers.aead import AESGCM

drive.mount("/content/drive")
LM_HOME = Path.home() / ".lmstudio"
VAULT = Path("/content/drive/MyDrive/LMStudio/colab-lmlink-auth.v1.enc")
VAULT_MAGIC = b"LMLINK-VAULT-1"
AUTH_DIRS = ("credentials", "hub")
MAX_AUTH_BYTES = 20 * 1024 * 1024

try:
    vault_password = userdata.get("LM_LINK_VAULT_PASSWORD")
except Exception:
    vault_password = None
if not vault_password:
    vault_password = getpass.getpass("LM Link認証保管庫のパスフレーズ: ")
    if not VAULT.exists():
        confirm = getpass.getpass("確認のため再入力: ")
        if vault_password != confirm:
            raise ValueError("パスフレーズが一致しないのじゃ")
if len(vault_password) < 12:
    raise ValueError("認証保管庫のパスフレーズは12文字以上を指定するのじゃ")

def vault_key(salt):
    return hashlib.pbkdf2_hmac("sha256", vault_password.encode("utf-8"), salt, 600000, 32)

def backup_login():
    payload = io.BytesIO()
    count = 0
    with zipfile.ZipFile(payload, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for directory in AUTH_DIRS:
            root = LM_HOME / directory
            if not root.is_dir() or root.is_symlink():
                continue
            for path in root.rglob("*"):
                if path.is_symlink() or not path.is_file():
                    continue
                if path.stat().st_size > MAX_AUTH_BYTES:
                    raise ValueError("認証ファイルのサイズが想定範囲外じゃ")
                z.write(path, path.relative_to(LM_HOME).as_posix())
                count += 1
    if not count:
        raise RuntimeError("LM Studioの認証ファイルが見つからず、保管庫は作成されなかった")
    if payload.tell() > MAX_AUTH_BYTES:
        raise ValueError("認証保管庫のサイズが想定範囲外じゃ")
    salt, nonce = secrets.token_bytes(16), secrets.token_bytes(12)
    encrypted = AESGCM(vault_key(salt)).encrypt(nonce, payload.getvalue(), VAULT_MAGIC)
    VAULT.parent.mkdir(parents=True, exist_ok=True)
    temp_path = VAULT.with_suffix(".tmp")
    with open(temp_path, "wb") as out:
        os.chmod(temp_path, 0o600)
        out.write(VAULT_MAGIC + salt + nonce + encrypted)
    os.replace(temp_path, VAULT)
    print("LM Link認証情報を暗号化してGoogleドライブに保存したのじゃ。")

def restore_login():
    if not VAULT.is_file():
        print("保存済み認証情報なし。初回ログイン後に暗号化して保存するのじゃ。")
        return False
    encrypted = VAULT.read_bytes()
    header_len = len(VAULT_MAGIC)
    if len(encrypted) < header_len + 16 + 12 + 16 or not encrypted.startswith(VAULT_MAGIC):
        raise ValueError("認証保管庫のファイル形式が不正じゃ")
    salt = encrypted[header_len:header_len + 16]
    nonce = encrypted[header_len + 16:header_len + 28]
    try:
        plaintext = AESGCM(vault_key(salt)).decrypt(nonce, encrypted[header_len + 28:], VAULT_MAGIC)
    except Exception as error:
        raise ValueError("認証保管庫を復号できない。パスフレーズまたはファイルを確認するのじゃ") from error
    if len(plaintext) > MAX_AUTH_BYTES:
        raise ValueError("認証保管庫のサイズが想定範囲外じゃ")
    with zipfile.ZipFile(io.BytesIO(plaintext)) as z:
        files = z.infolist()
        for info in files:
            parts = Path(info.filename).parts
            if (not parts or parts[0] not in AUTH_DIRS or
                    ".." in parts or Path(info.filename).is_absolute() or
                    info.is_dir() or info.file_size > MAX_AUTH_BYTES):
                raise ValueError("認証保管庫に不正なファイルが含まれるのじゃ")
        for info in files:
            destination = LM_HOME / info.filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src:
                with open(destination, "wb") as dst:
                    dst.write(src.read())
            os.chmod(destination, 0o600)
    print("暗号化したLM Link認証情報を復元したのじゃ。")
    return True

auth_restored = restore_login()

## 3. LM Linkのログインと接続

保存済み認証情報が有効なら再ログインせずに接続する。初回またはログイン期限切れなら `lms login` の認証URLをブラウザで開くのじゃ。ログイン後に認証情報を暗号化保存する。

In [ ]:
import subprocess
import time

subprocess.run(["lms", "daemon", "up"], check=True)
time.sleep(3)
whoami = subprocess.run(
    ["lms", "whoami"], stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL, check=False, timeout=30
)
if whoami.returncode != 0:
    print("LM Studioのログインが必要じゃ。表示される認証URLで承認するのじゃ。")
    subprocess.run(["lms", "login"], check=True)
else:
    print("保存済みのLM Studioログインが有効じゃ。")
subprocess.run(["lms", "link", "enable"], check=True)
backup_login()
print("LM Linkが有効になったのじゃ。")

## 4. PLaMo 3 Translateモデルのダウンロードとロード

0.8B・2B・8B・31Bから一つ選び、GGUFの量子化形式を指定する。`lms get`の検索処理を使わず、PLaMo公式のGGUFファイルを直接取得するため、問題のコマンドは実行しない。中断したダウンロードは途中から再開する。Colabランタイムを破棄するとモデルファイルは消えるが、Googleドライブの暗号化済み認証情報は残る。31BはGPUメモリ不足でロードできない場合があるのじゃ。

In [ ]:
import os
import subprocess
from pathlib import Path

plamo_size = "2b" # @param ["0.8b", "2b", "8b", "31b"]
quantization = "Q4_K_M" # @param ["Q4_K_M", "Q3_K_M", "Q2_K", "Q4_0"]

if quantization == "Q4_0" and plamo_size not in ("0.8b", "2b"):
    raise ValueError("Q4_0を選べるのは0.8Bと2Bだけじゃ")

repo_name = f"plamo-3-translate-{plamo_size}-GGUF"
filename = f"plamo-3-translate-{plamo_size}-{quantization}.gguf"
model_dir = Path.home() / ".lmstudio" / "models" / "pfnet" / repo_name
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / filename
partial_path = model_dir / (filename + ".part")
model_id = f"plamo-3-translate-{plamo_size}-colab"
url = f"https://huggingface.co/pfnet/{repo_name}/resolve/main/{filename}?download=true"

if not model_path.exists() or model_path.stat().st_size < 100_000_000:
    print(f"公式GGUFを取得: pfnet/{repo_name}/{filename}", flush=True)
    subprocess.run(
        ["curl", "--fail", "--location", "--retry", "3", "--retry-delay", "3",
         "--continue-at", "-", "--output", str(partial_path), url],
        check=True
    )
    if partial_path.stat().st_size < 100_000_000:
        raise RuntimeError("GGUFのダウンロード結果が小さすぎるのじゃ")
    os.replace(partial_path, model_path)
else:
    print(f"ダウンロード済みGGUFを再利用: {filename}", flush=True)

subprocess.run(["lms", "unload", "--all"], check=True)
subprocess.run(
    ["lms", "load", str(model_path), "--gpu", "max", "--identifier", model_id],
    check=True
)
subprocess.run(["lms", "server", "start", "--port", "1234"], check=True)
subprocess.run(["lms", "ps"], check=True)
print(f"ロード完了: {model_id}（LM Linkから利用可能）", flush=True)